# Sovereign AI: Fine-Tuning TinyAya on Yoruba for Constrained Kubernetes

This notebook trains a localized LoRA adapter on top of **CohereLabs/tiny-aya-earth** (3.35B parameters) using the **masakhane/african-ultrachat** (Yoruba split) dataset.

> **Tip:** Run on Google Colab with GPU enabled: **Runtime -> Change runtime type -> T4 GPU or A100**.

In [ ]:
# 1. Install rock-solid core dependencies (without fragile external wrappers)
!pip install -q -U "transformers>=4.44.0" "peft>=0.12.0" "datasets>=2.20.0" "accelerate>=0.33.0" bitsandbytes huggingface_hub

In [ ]:
# 2. Verify GPU allocation
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("WARNING: No GPU detected! Go to Runtime -> Change runtime type -> select T4 or A100 GPU.")

In [ ]:
# 3. Hugging Face Authentication
import os
from huggingface_hub import login

# Paste your Write-enabled Hugging Face token below
HF_TOKEN = input("Enter your Hugging Face Access Token: ").strip()
if HF_TOKEN:
    login(token=HF_TOKEN)
    os.environ["HF_TOKEN"] = HF_TOKEN
    print("Logged in to Hugging Face successfully!")
else:
    print("Warning: No token entered. If using gated models, access may fail.")

In [ ]:
# 4. Load Base Model and Run Baseline Evaluation (Before Training)
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

model_id = "CohereLabs/tiny-aya-earth"
compute_dtype = torch.bfloat16 if (torch.cuda.is_available() and torch.cuda.is_bf16_supported()) else torch.float16

# 4-bit Quantization Config for Colab GPU
if torch.cuda.is_available():
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=compute_dtype,
        bnb_4bit_use_double_quant=True,
    )
else:
    bnb_config = None

print(f"Loading base tokenizer and model: {model_id}...")
tokenizer = AutoTokenizer.from_pretrained(model_id, token=True, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

base_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto" if torch.cuda.is_available() else "cpu",
    torch_dtype=compute_dtype,
    token=True,
    trust_remote_code=True,
)

# Baseline Generation Test
test_prompt = "<|START_OF_TURN_TOKEN|><|USER_TOKEN|>Bawo ni o se le se alaye bi ero ayelujara (Internet) se n sise ni ede Yoruba to rorun?<|END_OF_TURN_TOKEN|><|START_OF_TURN_TOKEN|><|CHATBOT_TOKEN|>"
device = "cuda" if torch.cuda.is_available() else "cpu"
inputs = tokenizer(test_prompt, return_tensors="pt").to(device)

print("Generating baseline output...")
with torch.no_grad():
    outputs = base_model.generate(**inputs, max_new_tokens=150, temperature=0.7, do_sample=True)

baseline_response = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
print("\n--- BASELINE OUTPUT (Before Fine-Tuning) ---")
print(baseline_response)
print("-------------------------------------------\n")

In [ ]:
# 5. Prepare and Tokenize African-UltraChat Yoruba Dataset
from datasets import load_dataset

dataset_name = "masakhane/african-ultrachat"
print(f"Loading dataset: {dataset_name} (Yoruba split)...")

try:
    dataset = load_dataset(dataset_name, "yo", split="train")
except Exception as e:
    print(f"Direct split load fallback: {e}")
    dataset = load_dataset(dataset_name, split="train")
    if "language" in dataset.column_names:
        dataset = dataset.filter(lambda x: x["language"].lower() in ["yo", "yoruba"])

# Subsample 3000-4000 dialogues for efficient training (~30-45 mins on Colab GPU)
max_samples = min(3500, len(dataset))
dataset = dataset.shuffle(seed=42).select(range(max_samples))
print(f"Training set ready with {len(dataset)} conversation records.")

def format_and_tokenize(batch):
    texts = []
    for record_messages in batch["messages"]:
        turn_text = ""
        for msg in record_messages:
            role = msg.get("role", "")
            content = msg.get("content", "").strip()
            if role == "user":
                turn_text += f"<|START_OF_TURN_TOKEN|><|USER_TOKEN|>{content}<|END_OF_TURN_TOKEN|>"
            elif role == "assistant":
                turn_text += f"<|START_OF_TURN_TOKEN|><|CHATBOT_TOKEN|>{content}<|END_OF_TURN_TOKEN|>"
        texts.append(turn_text)
    return tokenizer(texts, truncation=True, max_length=1024, padding="max_length")

print("Tokenizing dataset...")
tokenized_dataset = dataset.map(format_and_tokenize, batched=True, remove_columns=dataset.column_names)
print("Dataset tokenization complete!")

In [ ]:
# 6. Configure LoRA and Train with standard Hugging Face Trainer
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import Trainer, TrainingArguments, DataCollatorForLanguageModeling

if torch.cuda.is_available():
    base_model = prepare_model_for_kbit_training(base_model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)

peft_model = get_peft_model(base_model, lora_config)
peft_model.print_trainable_parameters()

training_args = TrainingArguments(
    output_dir="./tiny-aya-earth-yoruba-lora",
    num_train_epochs=2,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    logging_steps=10,
    save_strategy="epoch",
    fp16=(compute_dtype == torch.float16),
    bf16=(compute_dtype == torch.bfloat16),
    max_grad_norm=0.3,
    warmup_steps=50,
    report_to="none",
)

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

trainer = Trainer(
    model=peft_model,
    train_dataset=tokenized_dataset,
    data_collator=data_collator,
    args=training_args,
)

print("Starting LoRA training on Yoruba conversations...")
trainer.train()

# Save LoRA adapter locally
trainer.model.save_pretrained("./tiny-aya-earth-yoruba-lora")
tokenizer.save_pretrained("./tiny-aya-earth-yoruba-lora")
print("Adapter successfully saved to ./tiny-aya-earth-yoruba-lora")

In [ ]:
# 7. Post-Training Evaluation: Test Identical Prompt
peft_model.eval()
print("Generating fine-tuned output with identical prompt...")
with torch.no_grad():
    outputs = peft_model.generate(**inputs, max_new_tokens=150, temperature=0.7, do_sample=True)

finetuned_response = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
print("\n--- FINE-TUNED OUTPUT (After LoRA Training) ---")
print(finetuned_response)
print("-----------------------------------------------\n")

In [ ]:
# 8. Push Adapter to Hugging Face Hub
hub_adapter_id = "husseinalamutu/tiny-aya-earth-yoruba-lora"
try:
    print(f"Pushing adapter to {hub_adapter_id}...")
    peft_model.push_to_hub(hub_adapter_id, token=True)
    tokenizer.push_to_hub(hub_adapter_id, token=True)
    print("Successfully uploaded adapter to Hugging Face!")
except Exception as e:
    print(f"Push to hub error: {e}")

In [ ]:
# 9. Merge LoRA Weights into Base Model and Save for GGUF Conversion
from peft import PeftModel
import gc

print("Consolidating weights: loading base model in FP16 on CPU...")
del base_model, peft_model, trainer
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

base_fp16 = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    device_map="cpu",
    token=True,
    trust_remote_code=True,
)

print("Loading trained LoRA adapter...")
merged_model = PeftModel.from_pretrained(base_fp16, "./tiny-aya-earth-yoruba-lora")
merged_model = merged_model.merge_and_unload()

merged_output_dir = "./tiny-aya-earth-yoruba-merged"
print(f"Saving merged weights to {merged_output_dir}...")
merged_model.save_pretrained(merged_output_dir)
tokenizer.save_pretrained(merged_output_dir)
print("Weight merge complete! Model is ready for GGUF quantization.")